# proto-REINFORCE on the Golden Goose, part 1: runs and cumulative rewards

This morning you wrote the transition dynamics of the Golden Goose. proto-REINFORCE learns its choice from runs, as it
learned to manage Old Bessie on page 3, in three notebooks: runs and cumulative rewards (this one), the forward pass,
the backward pass. Each notebook works without the others.

A goose lays a golden egg, worth \$10, in 80% of the weeks, and a butcher pays \$50 for it. Every week the
farmer chooses: KEEP the goose, or SELL it. A run is a quarter, 13 weeks, from GOOSE, and its cumulative reward is
the money of its 13 weeks added up. Here you build a run and its cumulative reward.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. The luck of the
goose is in the kit: your code never deals with dice.

Start with the folded cells below.

In [ ]:
#@title The goose's kit (run me first) { display-mode: "form" }
# The goose's kit. Run this cell first; you do not need to read it.
import inspect as _kit_inspect
import random                   # Python's random numbers: random.randint(0, 1) gives 0 or 1
import random as _kit_random
from math import exp            # e to the power: exp(0.5) is 1.6487...

GOOSE, SOLD = 0, 1              # the states
KEEP, SELL = 0, 1               # the actions
WEEKS = 13                      # a run lasts a quarter: 13 weeks
RUNS = 1000                     # an average takes 1,000 runs

_KIT_FIRST_LUCK = 1
_KIT_LUCK_BASE = 1000
_KIT_PICKS = 1
_KIT = {"seed": _KIT_FIRST_LUCK, "luck": _kit_random.Random(_KIT_FIRST_LUCK), "update": 0, "spins": 0,
        "forced": None}
_KIT_NAMES = {"states": ["GOOSE", "SOLD"], "actions": ["KEEP", "SELL"]}


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    _KIT["spins"] = _KIT["spins"] + 1
    if _KIT["forced"] is not None:
        return _KIT["forced"]
    return _KIT["luck"].random()


def transition(state, action):
    """One week of the goose. Gives the state next week and the week's money:
        next_state, money = transition(GOOSE, KEEP)"""
    ball = spin()
    if state == SOLD:
        return SOLD, 0                  # a sold goose brings nothing more, whatever the action
    if action == SELL:
        return SOLD, 50                 # the butcher pays $50, and the goose is SOLD
    if ball < 0.8:
        return GOOSE, 10                # a golden egg, $10, and the goose stays
    return GOOSE, 0                     # no egg, $0, and the goose stays


def softmax(bets):
    """A row of bets to a row of chances: e to the power of each bet, divided by their total."""
    largest = max(bets)
    powers = [exp(b - largest) for b in bets]
    total = sum(powers)
    return [p / total for p in powers]


def same_luck():
    """The luck starts again from its beginning: every average meets the same weeks."""
    _KIT["luck"] = _kit_random.Random(_KIT["seed"])


def new_luck():
    """New runs for the next update of proto-REINFORCE: the luck of seed 1000 plus the number of the update."""
    _KIT["update"] = _KIT["update"] + 1
    _KIT["seed"] = _KIT_LUCK_BASE + _KIT["update"]
    same_luck()


def start_loop():
    """The luck of the loop from its beginning, and the same picks of random.randint for everyone."""
    _kit_set_luck(_KIT_FIRST_LUCK)
    random.seed(_KIT_PICKS)


def _kit_set_luck(seed, update=0):
    _KIT["seed"] = seed
    _KIT["update"] = update
    same_luck()


def copy_bets(bets):
    """A copy of a row of bets: changing the copy leaves the bets as they are."""
    return list(bets)


def _kit_money(x):
    return ("$" if x >= 0 else "-$") + format(abs(x), ",.2f")


def plot_rewards(points, best=None):
    """The reward plot: the average G1 before the first update, then after each update."""
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 3.5))
    plt.plot(range(len(points)), points, color="#C8102E", linewidth=2, label="average G1 of your bets")
    if best is not None:
        plt.axhline(best, color="#333333", linestyle="--", linewidth=1, label="always KEEP")
    plt.xlabel("update")
    plt.ylabel("average G1 ($)")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()


def _kit_gaps(fn):
    """How many lines of the function still hold a gap, ..."""
    code_of = getattr(fn, "__code__", None)
    if code_of is None or Ellipsis not in code_of.co_consts:
        return 0
    try:
        src = _kit_inspect.getsource(fn)
    except (OSError, TypeError):
        return 1
    n = 0
    for line in src.splitlines():
        code_part = line.split("#")[0]
        if "..." in code_part:
            n = n + 1
    return n


def play(fn, *args):
    """Calls one of your functions once its check passes; otherwise prints one calm line."""
    for a in args:
        if a is None:
            print("⏳ %s waits for a value that an earlier cell has not given yet." % fn.__name__)
            return None
    try:
        ok = _KIT_CHECKS[fn.__name__](fn, quiet=True)
    except NameError:
        print("⏳ run the folded cell of the checks first.")
        return None
    except KeyError:
        return fn(*args)
    if not ok:
        print("⏳ %s is not ready yet: its check says what is missing." % fn.__name__)
        return None
    return fn(*args)


print("✅ The kit is ready: GOOSE, SOLD, KEEP, SELL, WEEKS, RUNS, spin, transition, softmax, same_luck, new_luck.")

In [ ]:
#@title The checks (run me too) { display-mode: "form" }
# The checks of the notebook. Run this cell too; you do not need to read it.
# A check prints one short message and never stops the notebook.

def _ref_make_roulette(chances):
    ends, running = [], 0
    for c in chances:
        running = running + c
        ends.append(running)
    return ends


def _ref_where_it_lands(ends, ball):
    for slot in range(len(ends)):
        if ball < ends[slot]:
            return slot
    return len(ends) - 1


def _ref_pick_action(chances):
    return _ref_where_it_lands(_ref_make_roulette(chances), spin())


def _ref_play_run(chances):
    rewards, state = [], GOOSE
    for week in range(WEEKS):
        action = _ref_pick_action(chances)
        state, money = transition(state, action)
        rewards.append(money)
    return rewards


def _ref_average_G1(bets):
    same_luck()
    chances = softmax(bets)
    total = 0
    for run in range(RUNS):
        total = total + sum(_ref_play_run(chances))
    return round(total / RUNS, 2)


def _ref_slope(bets, action, hop):
    before = _ref_average_G1(bets)
    hopped = list(bets)
    hopped[action] = hopped[action] + hop
    return round((_ref_average_G1(hopped) - before) / hop)


def _kit_try(name, fn, args, say):
    """Calls fn calmly: (True, result), or (False, None) after one line."""
    try:
        return True, fn(*args)
    except Exception as e:
        say("❌ %s stops with an error: %s: %s" % (name, type(e).__name__, e))
        return False, None


def _kit_ready(name, fn, task, say, needs=()):
    if not callable(fn):
        say("⏳ %s is not a function yet: run the cell of task %s first." % (name, task))
        return False
    gaps = _kit_gaps(fn)
    if gaps > 0:
        say("⏳ %s is not finished yet: replace every ... of task %s with your code (%d left)." % (name, task, gaps))
        return False
    for other in needs:
        f = globals().get(other)
        if f is None or not _KIT_CHECKS[other](f, quiet=True):
            say("⏳ %s calls %s: finish %s first, its check says what is missing." % (name, other, other))
            return False
    return True


def _kit_check_pick_action(pick_action, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("pick_action", pick_action, "A", say):
        return False
    for chances, ball, want in (([0.3, 0.7], 0.1, KEEP), ([0.3, 0.7], 0.5, SELL), ([0.9, 0.1], 0.89, KEEP),
                                ([0.9, 0.1], 0.95, SELL), ([0.3, 0.7], 0.3, SELL)):
        _KIT["forced"], _KIT["spins"] = ball, 0
        try:
            ok, got = _kit_try("pick_action(%s)" % chances, pick_action, (chances,), say)
        finally:
            _KIT["forced"] = None
        if not ok:
            return False
        if _KIT["spins"] != 1:
            say("❌ pick_action spins %d balls. It spins one: ball = spin(), once." % _KIT["spins"])
            return False
        if got != want:
            say("❌ pick_action(%s) with the ball at %s gives %r. The ends are %s, so the ball lands in %s's slot, %d."
                % (chances, ball, got, _ref_make_roulette(chances), _KIT_NAMES["actions"][want], want))
            return False
    say("✅ pick_action picks the slot where the ball lands in all 5 tests. Task A is done.")
    return True


def _kit_check_play_run(play_run, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("play_run", play_run, "B", say, ("pick_action",)):
        return False
    for chances, seed in (([0.5, 0.5], 7), ([0.9, 0.1], 8), ([1.0, 0.0], 9)):
        _kit_set_luck(seed)
        ok, got = _kit_try("play_run(%s)" % chances, play_run, (chances,), say)
        if not ok:
            return False
        _kit_set_luck(seed)
        want = _ref_play_run(chances)
        if not isinstance(got, list):
            say("❌ play_run gives %r. It gives the list of the run's rewards, one per week." % (got,))
            return False
        if len(got) != WEEKS:
            say("❌ play_run gives %d rewards. A run has WEEKS = 13 weeks, one reward each, also once the goose "
                "is SOLD." % len(got))
            return False
        if got != want:
            say("❌ play_run(%s) gives %s on the luck of the check; it should give %s. Each week: the action with "
                "pick_action(chances), then transition(state, action), from the state the week before left."
                % (chances, got, want))
            return False
    _kit_set_luck(_KIT_FIRST_LUCK)
    say("✅ play_run plays 13 weeks in all 3 tests. Task B is done.")
    return True


def _kit_check_cumulative_reward(cumulative_reward, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("cumulative_reward", cumulative_reward, "C", say):
        return False
    for rewards in ([10, 10, 0, 10, 50, 0, 0, 0, 0, 0, 0, 0, 0], [50] + [0] * 12, [10] * 13, [0, 10, 10]):
        ok, got = _kit_try("cumulative_reward", cumulative_reward, (rewards,), say)
        if not ok:
            return False
        if got != sum(rewards):
            say("❌ cumulative_reward(%s) gives %r. Every week counts in full: the total is %s."
                % (rewards, got, sum(rewards)))
            return False
    say("✅ cumulative_reward adds up the rewards in all 4 tests. Task C is done.")
    return True


def _kit_check_average_G1(average_G1, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("average_G1", average_G1, "D", say, ("play_run", "cumulative_reward")):
        return False
    for bets in ([0.0, 0.0], [2.0, -1.0]):
        _kit_set_luck(_KIT_FIRST_LUCK + 1)
        _KIT["luck"] = _kit_random.Random(12345)          # a luck the function must replace with same_luck()
        ok, got = _kit_try("average_G1(%s)" % bets, average_G1, (bets,), say)
        if not ok:
            return False
        want = _ref_average_G1(bets)
        if got != want:
            say("❌ average_G1(%s) gives %r; it should give %s. Its lines: same_luck(), the chances with "
                "softmax(bets), then RUNS runs, each one's G1 added to the total, and the total divided by RUNS."
                % (bets, got, want))
            _kit_set_luck(_KIT_FIRST_LUCK)
            return False
    _kit_set_luck(_KIT_FIRST_LUCK)
    say("✅ average_G1 gives the average G1 of 1,000 runs in both tests. Task D is done.")
    return True


def _kit_check_pick_bet(pick_bet, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("pick_bet", pick_bet, "E", say):
        return False
    saved = random.getstate()
    seen = set()
    try:
        for i in range(200):
            ok, got = _kit_try("pick_bet()", pick_bet, (), say)
            if not ok:
                return False
            if got not in (0, 1):
                say("❌ pick_bet() gives %r. It gives the action of a bet: 0 for KEEP or 1 for SELL." % (got,))
                return False
            seen.add(got)
    finally:
        random.setstate(saved)
    if seen != {0, 1}:
        say("❌ pick_bet() gave only %s in 200 picks. It picks KEEP or SELL at random: random.randint(0, 1)."
            % _KIT_NAMES["actions"][seen.pop()])
        return False
    say("✅ pick_bet picks KEEP and SELL. Task E is done.")
    return True


def _kit_check_slope(slope, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("slope", slope, "F", say, ("average_G1",)):
        return False
    for bets, action in (([-0.5, 0.5], KEEP), ([-0.5, 0.5], SELL), ([1.0, 0.0], KEEP)):
        before = list(bets)
        _kit_set_luck(_KIT_FIRST_LUCK)
        ok, got = _kit_try("slope(%s, %s, 0.01)" % (bets, _KIT_NAMES["actions"][action]), slope,
                           (bets, action, 0.01), say)
        if not ok:
            return False
        if bets != before:
            say("❌ slope changed the bets themselves, to %s. The hop goes into a copy: hopped = copy_bets(bets)."
                % bets)
            return False
        _kit_set_luck(_KIT_FIRST_LUCK)
        want = _ref_slope(bets, action, 0.01)
        if got != want:
            say("❌ slope(%s, %s, 0.01) gives %r; it should give %s: Before, After with the bet a hop higher in a "
                "copy, then (After - Before) / hop, rounded." % (bets, _KIT_NAMES["actions"][action], got, want))
            return False
    _kit_set_luck(_KIT_FIRST_LUCK)
    say("✅ slope gives nabla in all 3 tests. Task F is done.")
    return True


def _kit_check_update_bet(update_bet, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("update_bet", update_bet, "G", say):
        return False
    for bets, action, nabla, alpha in (([-0.5, 0.5], KEEP, 37, 0.01), ([-0.5, 0.5], SELL, -12, 0.01),
                                       ([1.0, 2.0], KEEP, 0, 0.01)):
        b = list(bets)
        ok, got = _kit_try("update_bet", update_bet, (b, action, nabla, alpha), say)
        if not ok:
            return False
        want = round(bets[action] + alpha * nabla, 2)
        if b[action] != want:
            say("❌ update_bet(%s, %s, %s, %s) leaves the bet at %r; it should be %s: w + alpha * nabla."
                % (bets, _KIT_NAMES["actions"][action], nabla, alpha, b[action], want))
            return False
    say("✅ update_bet moves the bet in all 3 tests. Task G is done.")
    return True


def _kit_check_proto_reinforce(proto_reinforce, quiet=False):
    say = (lambda line: None) if quiet else print
    if not _kit_ready("proto_reinforce", proto_reinforce, "H", say, ("pick_bet", "slope", "update_bet")):
        return False
    bets = [-0.5, 0.5]
    start_loop()
    ok, got = _kit_try("proto_reinforce", proto_reinforce, (bets, 3, 0.01, 0.01), say)
    if not ok:
        return False
    start_loop()
    want_bets = [-0.5, 0.5]
    want = [_ref_average_G1(want_bets)]
    for u in range(3):
        new_luck()
        action = random.randint(0, 1)
        nabla = _ref_slope(want_bets, action, 0.01)
        want_bets[action] = round(want_bets[action] + 0.01 * nabla, 2)
        want.append(_ref_average_G1(want_bets))
    start_loop()
    _kit_set_luck(_KIT_FIRST_LUCK)
    if not isinstance(got, list) or len(got) != 4:
        say("❌ proto_reinforce(bets, 3, ...) gives %r. It gives the points of the reward plot: the first point, "
            "then one after each of the 3 updates, 4 in all." % (got,))
        return False
    if bets != want_bets or got != want:
        say("❌ 3 updates from [-0.5, 0.5] give the bets %s and the points %s; they should give %s and %s. Each "
            "update: new_luck(), pick_bet(), slope, update_bet, then the point average_G1(bets)."
            % (bets, got, want_bets, want))
        return False
    say("✅ proto_reinforce plays 3 updates as it should. Task H is done.")
    return True


_KIT_CHECKS = {
    "pick_action": _kit_check_pick_action,
    "play_run": _kit_check_play_run,
    "cumulative_reward": _kit_check_cumulative_reward,
    "average_G1": _kit_check_average_G1,
    "pick_bet": _kit_check_pick_bet,
    "slope": _kit_check_slope,
    "update_bet": _kit_check_update_bet,
    "proto_reinforce": _kit_check_proto_reinforce,
}


def check_pick_action(pick_action):
    _kit_check_pick_action(pick_action)


def check_play_run(play_run):
    _kit_check_play_run(play_run)


def check_cumulative_reward(cumulative_reward):
    _kit_check_cumulative_reward(cumulative_reward)


def check_average_G1(average_G1):
    _kit_check_average_G1(average_G1)


def check_pick_bet(pick_bet):
    _kit_check_pick_bet(pick_bet)


def check_slope(slope):
    _kit_check_slope(slope)


def check_update_bet(update_bet):
    _kit_check_update_bet(update_bet)


def check_proto_reinforce(proto_reinforce):
    _kit_check_proto_reinforce(proto_reinforce)


print("✅ The checks are ready.")

## 1. The goose

Two states, GOOSE and SOLD, and two actions, KEEP and SELL. The kit holds this morning's transition dynamics as
`transition(state, action)`; it gives back the state next week and the week's money:

- GOOSE and KEEP: a golden egg, \$10, in 80% of the weeks, else \$0; the goose stays.
- GOOSE and SELL: the butcher pays \$50; the goose is SOLD.
- SOLD and either action: \$0; the goose stays SOLD.

Run the next cell a few times.

In [ ]:
next_state, money = transition(GOOSE, KEEP)
print("GOOSE, KEEP:   next state", ["GOOSE", "SOLD"][next_state], "   money", money)
next_state, money = transition(GOOSE, SELL)
print("GOOSE, SELL:   next state", ["GOOSE", "SOLD"][next_state], "   money", money)

## 2. A strategy and its roulette

Only GOOSE has a choice: in SOLD nothing happens, whatever the action. So a strategy is one row of chances,
`[KEEP's chance, SELL's chance]`. The strategy `[0.3, 0.7]` keeps the goose in 30% of the weeks.

A roulette picks the action: a wheel from 0 to 1, cut into one slot per action, each as long as its chance. For
`[0.3, 0.7]`, KEEP from 0 to 0.3 and SELL from 0.3 to 1. A ball lands somewhere between 0 and 1, every place as
likely as any other, and its slot is the action.

### The roulette in Python, given

- `make_roulette(chances)` gives the ends of the slots, a running total of the chances: `[0.3, 0.7]` gives
  `[0.3, 1.0]`.
- `where_it_lands(ends, ball)` gives the first slot whose end is above the ball: 0 for KEEP, 1 for SELL.
- The kit's `spin()` throws the ball: a number between 0 and 1.

Run the next cell.

In [ ]:
# Given: the ends of the slots of a roulette
def make_roulette(chances):
    # chances: a row of chances, one per action. Gives the ends of the slots, a running total: [0.3, 1.0].
    ends = []
    running = 0                         # where the slots so far end: nowhere yet
    for chance in chances:
        running = running + chance      # this slot ends one chance further
        ends.append(running)            # remember where this slot ends
    return ends


# Given: the slot where the ball lands
def where_it_lands(ends, ball):
    # ends: the ends of the slots, from make_roulette. ball: a number between 0 and 1.
    # Gives the number of the first slot whose end is above the ball: 0 for KEEP, 1 for SELL.
    for slot in range(len(ends)):
        if ball < ends[slot]:
            return slot
    return len(ends) - 1                # a ball at the very end lands in the last slot


print("✅ make_roulette and where_it_lands are ready.")

In [ ]:
ends = make_roulette([0.3, 0.7])
print("the ends of the slots:", ends)
print("a ball at 0.25 lands in slot", where_it_lands(ends, 0.25))
print("a ball at 0.75 lands in slot", where_it_lands(ends, 0.75))

**Reading the result.** 0.25 is below 0.3, KEEP's end: slot 0, KEEP. 0.75 is not below 0.3 and below 1: slot 1,
SELL.

### 🎯 Task A. An action at random

Write `pick_action(chances)` in three steps, one line each:

1. The roulette: `make_roulette(chances)` gives the ends of its slots.
2. The ball: `spin()`, already in the skeleton.
3. The action: `where_it_lands(ends, ball)`.

In [ ]:
# 🎯 YOUR TURN, task A: an action at random, with the chances of the strategy
def pick_action(chances):
    # chances: the strategy, a row of chances [KEEP, SELL]. Gives KEEP (0) or SELL (1), picked at random.
    ends = ...                          # 🎯 YOUR TURN: the roulette of the chances, with make_roulette
    ball = spin()                       # the ball
    return ...                          # 🎯 YOUR TURN: where the ball lands, with where_it_lands

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# ends = make_roulette(...) with the strategy's chances.
# return where_it_lands(...) with the ends and the ball.

In [ ]:
#@title 💡 Hint 2 for task A (double-click to open) { display-mode: "form" }
#     ends = make_roulette(chances)
#     return where_it_lands(ends, ball)

In [ ]:
check_pick_action(pick_action)

## 3. A run

A run is a quarter, `WEEKS` = 13 weeks, from GOOSE. Each week the strategy picks an action, and the goose's week
gives the next state and the money. Once the goose is SOLD the weeks go on, each with \$0.

### 🎯 Task B. One run

Write `play_run(chances)`: each week, the action with your `pick_action`, then `transition(state, action)`, then
the week's money at the end of `rewards`.

In [ ]:
# 🎯 YOUR TURN, task B: one run of the goose
def play_run(chances):
    # chances: the strategy. Plays WEEKS weeks from GOOSE and gives the rewards: the money of every week, in a list.
    rewards = []
    state = GOOSE
    for week in range(WEEKS):
        action = ...                    # 🎯 YOUR TURN: the strategy picks an action
        state, money = ...              # 🎯 YOUR TURN: the goose's week
        rewards.append(...)             # 🎯 YOUR TURN: remember the week's money
    return rewards

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# Two names on the left of = catch the two things transition gives back: state, money = ...
# Each week starts from the state the week before left: transition(state, action).

In [ ]:
#@title 💡 Hint 2 for task B (double-click to open) { display-mode: "form" }
#         action = pick_action(chances)
#         state, money = transition(state, action)
#         rewards.append(money)

In [ ]:
check_play_run(play_run)

💭 **Think first.** The strategy `[0.9, 0.1]` keeps the goose in 9 weeks of 10. In which week do you expect the
butcher to come? Run the next cell a few times.

In [ ]:
rewards = play(play_run, [0.9, 0.1])
if rewards:
    print(rewards)

**Reading the result.** \$10 for an egg, \$0 for a week without one, \$50 the week the goose is sold, and \$0
every week after it. Some runs never meet the butcher: 0.9 to the power 13 is 0.25.

## 4. The cumulative reward

The cumulative reward $G_1$ of a run is its money from week 1 on. On the goose every week counts in full, as the
money of a try this morning:

$$G_1 = r_1 + r_2 + \dots + r_{13}$$

### 🎯 Task C. The cumulative reward

Write `cumulative_reward(rewards)`: the rewards added up, one at a time.

In [ ]:
# 🎯 YOUR TURN, task C: the cumulative reward of a run
def cumulative_reward(rewards):
    # rewards: the money of every week of a run. Gives G1: every week's money counts in full.
    G1 = 0
    for money in rewards:
        G1 = ...                        # 🎯 YOUR TURN: add the week's money
    return G1

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# The loop sees one week's money at a time, money: add it to G1.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
#         G1 = G1 + money

In [ ]:
check_cumulative_reward(cumulative_reward)

In [ ]:
rewards = play(play_run, [0.9, 0.1])
if rewards:
    print(rewards, "  G1 =", play(cumulative_reward, rewards))

## What you built

- **A strategy's action**: `pick_action` on the given roulette, `make_roulette` and `where_it_lands`.
- **A run**: `play_run`, 13 weeks from GOOSE, the money of every week.
- **The cumulative reward**: `cumulative_reward`, the money of the weeks added up.

Next, part 2 builds the forward pass: [proto-REINFORCE on the Golden Goose, part 2: the forward pass](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/05_cx_golden_goose_2_forward_pass_student.ipynb).